In [1]:
import os
from sklearn.model_selection import train_test_split
import pandas as pd

# 1. Define the base directory where the image classes are located
base_dir = '../dataset'
# 2. Gather all image paths and their corresponding labels
image_paths = []
labels = []

for class_name in os.listdir(base_dir):
    class_dir = os.path.join(base_dir, class_name)
    if os.path.isdir(class_dir):
        for image_file in os.listdir(class_dir):
            if image_file.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp')):
                image_paths.append(os.path.join(class_dir, image_file))
                labels.append(class_name)

print(f"Total images found: {len(image_paths)}")
print(f"Total labels found: {len(labels)}")
print(f"Unique classes: {len(set(labels))}")

Total images found: 2000
Total labels found: 2000
Unique classes: 5


In [4]:
# Split the dataset
import numpy as np

# Split data into training (70%) and a temporary set (30% for validation + test)
X_train, X_temp, y_train, y_temp = train_test_split(image_paths, labels, test_size=0.30, random_state=42, stratify=labels)

# Split the temporary set into validation (15%) and test (15%)
# Since X_temp is 30% of the total, 0.5 of X_temp will be 15% of the total
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

print(f"Number of images in training set: {len(X_train)}")
print(f"Number of images in validation set: {len(X_val)}")
print(f"Number of images in test set: {len(X_test)}")

Number of images in training set: 1400
Number of images in validation set: 300
Number of images in test set: 300


Verify class distribution in the training, validation and test sets to confirm
that stratification is successful.

In [5]:
from collections import Counter

print("\n--- Training Set Class Distribution ---")
train_class_distribution = Counter(y_train)
for class_name, count in train_class_distribution.items():
    print(f"  {class_name}: {count} images ({count / len(y_train) * 100:.2f}%) ")

print("\n--- Validation Set Class Distribution ---")
val_class_distribution = Counter(y_val)
for class_name, count in val_class_distribution.items():
    print(f"  {class_name}: {count} images ({count / len(y_val) * 100:.2f}%) ")

print("\n--- Test Set Class Distribution ---")
test_class_distribution = Counter(y_test)
for class_name, count in test_class_distribution.items():
    print(f"  {class_name}: {count} images ({count / len(y_test) * 100:.2f}%) ")


--- Training Set Class Distribution ---
  Downy Mildew: 280 images (20.00%) 
  Healthy Leaf: 280 images (20.00%) 
  Bacterial Leaf Spot: 280 images (20.00%) 
  Mosaic Disease: 280 images (20.00%) 
  Powdery_Mildew: 280 images (20.00%) 

--- Validation Set Class Distribution ---
  Healthy Leaf: 60 images (20.00%) 
  Downy Mildew: 60 images (20.00%) 
  Bacterial Leaf Spot: 60 images (20.00%) 
  Mosaic Disease: 60 images (20.00%) 
  Powdery_Mildew: 60 images (20.00%) 

--- Test Set Class Distribution ---
  Bacterial Leaf Spot: 60 images (20.00%) 
  Downy Mildew: 60 images (20.00%) 
  Healthy Leaf: 60 images (20.00%) 
  Mosaic Disease: 60 images (20.00%) 
  Powdery_Mildew: 60 images (20.00%) 


In [ ]:
import os

# 1. Define the base path for the new dataset structure
output_base_dir = '../processed_data'

# 2. Define the names for the training, validation, and test directories
train_dir = os.path.join(output_base_dir, 'train')
val_dir = os.path.join(output_base_dir, 'val')
test_dir = os.path.join(output_base_dir, 'test')

# List of all main split directories
split_dirs = [train_dir, val_dir, test_dir]

# 3. Get the list of unique class names from your `labels` variable
unique_classes = sorted(list(set(labels)))

print(f"Creating base output directories: {output_base_dir}")

# 4. Create the main training, validation, and test directories
for s_dir in split_dirs:
    os.makedirs(s_dir, exist_ok=True)
    print(f"  Created directory: {s_dir}")

# 5. For each of the main directories, iterate through the unique class names
# and create a subdirectory for each class within them
print("Creating class subdirectories within train, val, and test splits...")
for s_dir in split_dirs:
    for class_name in unique_classes:
        class_path = os.path.join(s_dir, class_name)
        os.makedirs(class_path, exist_ok=True)
        print(f"  Created class directory: {class_path}")

print("Directory structure for processed data created successfully.")

Creating base output directories: ../processed_data
  Created directory: ../processed_data/train
  Created directory: ../processed_data/val
  Created directory: ../processed_data/test
Creating class subdirectories within train, val, and test splits...
  Created class directory: ../processed_data/train/Bacterial Leaf Spot
  Created class directory: ../processed_data/train/Downy Mildew
  Created class directory: ../processed_data/train/Healthy Leaf
  Created class directory: ../processed_data/train/Mosaic Disease
  Created class directory: ../processed_data/train/Powdery_Mildew
  Created class directory: ../processed_data/val/Bacterial Leaf Spot
  Created class directory: ../processed_data/val/Downy Mildew
  Created class directory: ../processed_data/val/Healthy Leaf
  Created class directory: ../processed_data/val/Mosaic Disease
  Created class directory: ../processed_data/val/Powdery_Mildew
  Created class directory: ../processed_data/test/Bacterial Leaf Spot
  Created class directory:

In [41]:
import os
import shutil

# Helper function to copy images to their respective split directories
def copy_images(image_paths, target_dir):
    print(f"Copying {len(image_paths)} images to {target_dir}...")
    for img_path in image_paths:
        # Extract class name from the image path
        class_name = os.path.basename(os.path.dirname(img_path))
        
        # Construct destination directory and path
        destination_class_dir = os.path.join(target_dir, class_name)
        destination_path = os.path.join(destination_class_dir, os.path.basename(img_path))
        
        # Copy the image file
        try:
            shutil.copy(img_path, destination_path)
        except FileNotFoundError:
            print(f"Warning: Source file not found: {img_path}")
        except Exception as e:
            print(f"Error copying {img_path} to {destination_path}: {e}")

# Define the target directories (already created in previous step)
output_base_dir = '../processed_data'
train_dir = os.path.join(output_base_dir, 'train')
val_dir = os.path.join(output_base_dir, 'val')
test_dir = os.path.join(output_base_dir, 'test')

# Copy images for each split
copy_images(X_train, train_dir)
copy_images(X_val, val_dir)
copy_images(X_test, test_dir)

print("Image copying process completed for all splits.")

Copying 1400 images to ../processed_data/train...
Copying 300 images to ../processed_data/val...
Copying 300 images to ../processed_data/test...
Image copying process completed for all splits.


In [48]:
## Preprocess train dataset
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# 1. Define target image size and batch size
IMG_HEIGHT = 224
IMG_WIDTH = 224
BATCH_SIZE = 32

# 2. Instantiate ImageDataGenerator with augmentation and preprocessing parameters
train_datagen = ImageDataGenerator(
    rescale=1./255, # Normalize pixel values to [0, 1]
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest'
)

# 3. Create the training data generator
train_generator = train_datagen.flow_from_directory(
    '../processed_data/train', # Directory containing training images
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)

print("Training data generator 'train_generator' created successfully with augmentation and normalization.")

Found 1400 images belonging to 5 classes.
Training data generator 'train_generator' created successfully with augmentation and normalization.


In [50]:
# Preprocess validation and test datasets
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# 1. Instantiate ImageDataGenerator for validation and test sets (no augmentation)
val_test_datagen = ImageDataGenerator(
    rescale=1./255 # Normalize pixel values to [0, 1]
)

# 2. Create the validation data generator
validation_generator = val_test_datagen.flow_from_directory(
    '../processed_data/val', # Directory containing validation images
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False # Do not shuffle validation data
)

# 3. Create the test data generator
test_generator = val_test_datagen.flow_from_directory(
    '../processed_data/test', # Directory containing test images
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False # Do not shuffle test data
)

print("Validation and test data generators created successfully with normalization and resizing.")

Found 300 images belonging to 5 classes.
Found 300 images belonging to 5 classes.
Validation and test data generators created successfully with normalization and resizing.


In [ ]:
# Create data loaders for training, validation, and test sets